# Custom Leaf Handler

This is an advance topic.  Make sure you are familar with [checkpointing_pytrees.ipynb](checkpointing_pytrees.ipynb) before reading this notebook.

PyTrees are a common tree structure used to represent training states. LeafHandlers are responsible for serializing and deserializing each leaf node. Different leaf object types require specific LeafHandlers. Orbax includes standard LeafHandlers for common types including jax.Array, np.ndarray, int, float, and str. Before creating a custom LeafHandler, always check the options available in ocp.options.PytreeOptions to ensure no existing options can meet your needs.


One of common reasons to have a custom LeafHandler is to support a custom type that is not supported by Orbax.  I will use the `Point` class from [customization.ipynb](customization.ipynb) as the example.  Let's say you need to checkpoint many Point objects in a nested tree structure.  It might make sense to store it within a Pytree along with your train state.  Then you would need to write a PointLeafHandler and register it with the LeafHandlerRegistry.

In [25]:
import dataclasses
import json
from typing import Awaitable, Type
from etils import epath
import numpy as np
from orbax.checkpoint import multihost
import orbax.checkpoint.experimental.v1 as ocp


@dataclasses.dataclass
class Point:
  x: int
  y: float

For LeafHandler, we need to define a AbtractPoint class as well.  This is required for two reasons:
1. The AbstractPoint class is used during restoration to indicate what type of a leaf object will be restored as.
2. In addition, metadata of a leaf node will be returned as AbstractPoint, avoid the need to restore the actual leaf object.

In following example of AbstractPoint, we just define it as the type of data members without actual values.

In [26]:
@dataclasses.dataclass
class AbstractPoint:
  x: Type[int] = int
  y: Type[float] = float

Next we will define the actual PointLeafHandler.  See the comments below which explain what functions are required.

In [27]:
from typing import Sequence
import asyncio

In [8]:
class PointLeafHandler(ocp.LeafHandler[Point, AbstractPoint]):
  """A custom leaf handler for testing."""

  def __init__(self, context: ocp.Context | None = None):
    """Required Initializer.

    This initializer is initialized lazily during checkpoint operations.  If the
    signature is not matched, an exception will be raised during initialization.

    Args:
      context: The context for the leaf handler.  The leaf handler can
        initialize and operate according to the context.  In this example, we do
        not utilize it though.  For more examples, see ArrayLeafHandler.
    """
    del context

  async def serialize(
      self,
      params: Sequence[ocp.SerializationParam[Point]],
      serialization_context: ocp.SerializationContext,
  ) -> Awaitable[None]:
    """Required Serialize function.

    This function writes the specified leaves of a checkpointable to a storage
    location.  A couple of notes here:
    1. This function is called on all hosts, but in this example, only the
    primary host w write.
    2. we use `await await_creation()` to ensure the parent directory is created
    before writing.
    """

    # make sure the parent directory is created
    await serialization_context.parent_dir.await_creation()

    def _background_serialize(params, serialization_context):

      # only the primary host writes
      if multihost.is_primary_host(0):
        for param in params:
          with open(
              serialization_context.parent_dir.path / f'{param.name}.txt',
              'w',
          ) as f:
            f.write(json.dumps(dataclasses.asdict(param.value)))

    return asyncio.to_thread(
        _background_serialize, params, serialization_context
    )

  async def deserialize(
      self,
      params: Sequence[ocp.DeserializationParam[AbstractPoint]],
      deserialization_context: ocp.DeserializationContext,
  ) -> Awaitable[Sequence[Point]]:
    """Required Deserialize function.

    Returns sequence of leaves from a stored checkpointable location. Note that
    we use asyncio.to_thread to ensure the deserialization is performed in a
    background thread immediately before returning this call.
    """

    def _deserialize_impl():
      ret = []
      for param in params:
        with open(
            deserialization_context.parent_dir / f'{param.name}.txt',
            'r',
        ) as f:
          ret.append(Point(**json.loads(f.read())))

      return ret

    return asyncio.to_thread(_deserialize_impl)

  async def metadata(
      self,
      params: Sequence[ocp.DeserializationParam[None]],
      deserialization_context: ocp.DeserializationContext,
  ) -> Sequence[AbstractPoint]:
    """Required Metadata function.

    Returns a sequence of metadata that helps to describe the available leaves
    in this checkpoint location.
    """

    return [AbstractPoint()] * len(params)

Next, we will define a train_state for demostration purpose.  In this train_state, it has some common types as well as some Points that are nested inside the PyTree.

In [31]:
# define a PyTree Train State

train_state = {
    'a': np.arange(16),
    'b': np.ones(16),
    'scalar': 123.0,
    'mixed': {
        'a': np.arange(16),
        'b': np.ones(16),
        'scalar': 123.0,
        'Point': Point(0, 0.5),
    },
    'Points': {
        'level1': {
            'a': Point(1, 2),
            'b': Point(3, 4),
            'level2': {'a': Point(5, 6), 'b': Point(7, 8), 'c': Point(9, 10)},
        }
    },
}

Next, we will prepare a LeafHandlerRegistry.  In this registry, the type and its abstract type will map with a LeafHandler.  In the following example, we create a `StandardLeafHadnler` first.  This is the same as the registry used by default.  Then PointLeafHandler is added along its type Point and abstract type AbstractPoint.  Note that only the `PointLeafHandler` type is registered, not the handler instance.  The instance will be created lazily depending on checkpoint operations.

In [32]:
# Create LeafHandlerRegisry
registry = ocp.StandardLeafHandlerRegistry() # with standard handlers
registry.add(Point, AbstractPoint, PointLeafHandler) # add custom handler

In [33]:
# prepare the checkpoint directory
path = epath.Path('/tmp/with_points')
path.rmtree(missing_ok=True)

Now, we are ready to save the `train_state`.  To customize context and pass the custom registry, you can use the `ocp.Context` as below.

In [34]:
with ocp.Context(
    pytree_options=ocp.options.PyTreeOptions(
            leaf_handler_registry=registry
    )
):
  ocp.save_pytree(path, train_state)

After saving, let's load the checkpoint back to see if we can get back the expected Point objects.  We will again create a ocp.Context with our custom registry.

In [35]:
with ocp.Context(
    pytree_options=ocp.options.PyTreeOptions(
            leaf_handler_registry=registry
    )
):
  restored_train_state = ocp.load_pytree(path)

In [36]:
import pprint
pprint.pprint(restored_train_state)

We can see the restored_train_state looks exactly the same as the original train_state.

Finally, we also want to see if we can read the expected metadata.  Similarly, we will use ocp.Context to use our registry with the custom PointLeafHandler.

In [37]:
with ocp.Context(
    pytree_options=ocp.options.PyTreeOptions(
            leaf_handler_registry=registry
    )
):
  restored_metadata = ocp.pytree_metadata(path)

We can see the AbstractPoints are returned for Point leaves.

In [38]:
pprint.pprint(restored_metadata.metadata)